# Lista 6 De Computacão Quântica
## Paulo Vinicius Pretto 304450

#### Para rodar este codigo, antes rode no seu terminal linux os seguintes comandos:
*se estiver em um SO windows, instala o **wsl** e pode rodar tranquilamente também*

`` python3 -m venv env  `` ( *env* pode ser um nome qualquer para o seu ambiente virtual)  
  
`` source env/bin/activate ``  
  
`` pip install -r requeriments.txt ``

In [4]:
from qiskit import *
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
import numpy as np
import random as rnd

In [22]:
def dj_oracle(n, oracle):
   
    # O oráculo opera em n qubits de entrada + 1 qbit alvo (ancilla)
    qc = QuantumCircuit(n + 1, name=f"Oracle ({oracle})")
    
    if oracle == "c":
        output = rnd.randint(0, 1)
        if output == 1:
            qc.x(n)
        
    elif oracle == "b":
        b_str = bin(rnd.randint(0, 2**n - 1))[2:].zfill(n)
        

        for qbit in range(n):
            if b_str[qbit] == '1':
                qc.x(qbit)

        for qbit in range(n):
            qc.cx(qbit, n)
            
        for qbit in range(n):
            if b_str[qbit] == '1':
                qc.x(qbit)
    else:
        raise ValueError("oracle deve ser 'c' para constante ou 'b' para balanceado")

    return qc

In [ ]:
def run_deutsch_jozsa(n, oracle):
    """
    Constrói e executa o algoritmo de Deutsch-Jozsa.
    Simula a nossa ignorância recebendo o oráculo como uma 'caixa preta'.
    """
    print("\n" + "-"*60)
    print(f"Deutsch-Jozsa para n= {n} e tipo= {oracle}")
    
    # 1. Obter o oráculo (Simulação da caixa preta)
    oracle_gate = dj_oracle(n, oracle)
    
    # 2. Construção do Circuito Principal
    # n qubits de dados + 1 ancilla
    qc = QuantumCircuit(n + 1, n)
    
    # Preparação do estado inicial
    # Qubits de entrada em |0>, Ancilla em |1> (para o phase kickback)
    qc.x(n) 
    
    # Aplica Hadamard em TODOS os qubits (entradas e ancilla)
    for i in range(n + 1):
        qc.h(i)
        
    qc.barrier()
    
    # 3. Aplicação do Oráculo (Uf)
    qc.append(oracle_gate, range(n + 1))
    
    qc.barrier()
    
    # 4. Hadamard final apenas nos qubits de entrada (interferência)
    for i in range(n):
        qc.h(i)
        
    # 5. Medição dos n qubits de entrada
    for i in range(n):
        qc.measure(i, i)
        
    # 6. Simulação
    simulator = AerSimulator()
    t_qc = transpile(qc, simulator)
    result = simulator.run(t_qc, shots=1000).result()
    counts = result.get_counts()
    
    # 7. Interpretação dos Resultados
    # Se medirmos '00...0' (apenas zeros), a função é CONSTANTE.
    # Qualquer outro resultado indica que a função é BALANCEADA.
    
    print(f"Contagens de medição: {counts}")
    
    # Verificar se o estado '0'*n (ex: '00' ou '000') está presente com ~100% de probabilidade
    zero_state = '0' * n
    if zero_state in counts and counts[zero_state] == 1000:
        interpretacao = "c"
    else:
        interpretacao = "b"
        
    print(f"Interpretação : {interpretacao}")
    
    # Validação
    if interpretacao.lower() == oracle:
        print("O algoritmo identificou corretamente o caso.")
    else:
        print("aaaaaaaaa -? Identificação incorreta.")
        
    return qc

In [24]:
# Executando para n=2 (Todos os tipos)
run_deutsch_jozsa(n=2, oracle="c")
run_deutsch_jozsa(n=2, oracle="b")

# Executando para n=3 (Todos os tipos)
run_deutsch_jozsa(n=3, oracle="c")
run_deutsch_jozsa(n=3, oracle="b")


------------------------------------------------------------
Deutsch-Jozsa para n= 2 e tipo= c
Contagens de medição: {'00': 1000}
Interpretação : C
O algoritmo identificou corretamente o caso.

------------------------------------------------------------
Deutsch-Jozsa para n= 2 e tipo= b
Contagens de medição: {'11': 1000}
Interpretação : B
O algoritmo identificou corretamente o caso.

------------------------------------------------------------
Deutsch-Jozsa para n= 3 e tipo= c
Contagens de medição: {'000': 1000}
Interpretação : C
O algoritmo identificou corretamente o caso.

------------------------------------------------------------
Deutsch-Jozsa para n= 3 e tipo= b
Contagens de medição: {'111': 1000}
Interpretação : B
O algoritmo identificou corretamente o caso.
